### Toolformer: Language Models Can Teach Themselves to Use Tools_2023
- 요약 : 현재 존재하는 Tool-Calling-Agent는 어떤 도구를 호출할 지 추가적인 LLM 사용이 필요하다는 단점이 존재한다. 이 논문에서 제시한 Toolformer는 GPT-J를 기반으로 자체적인 in-context 학습을 한 모델로 input query를 받으면 모델이 어느 위치에서 어떤 API의 호출이 필요한지를 스스로 판단하여 API를 호출, 그 결과를 이용하여 zero-shot으로 답변을 완성하는 방식을 구현해내었다.

- 방법론 : 이는 파인튜닝이 LM으로 하여금 API호출이 필요하다고 생각되는 곳과 그때 사용할 API를 판단하도록 한 뒤(Sampling API Calls) 그 호출 결과를 받아보고(Executing API Calls) 이를 토대로 더 나은 답변이 생성되었는지(Filtering API Calls) 평가하는 과정으로 이루어졌기 때문에 가능하였다. 해당 평가는 이전 토큰들(문맥)과 추가 정보를 받고 예측한 다음 토큰의 NLL값에 API 호출 토큰과의 거리 weight 값의 곱의 합으로 이루어졌다. `"<API>"`, `"</API>"`, `"->"` 등의 별도로 정의된 special token을 할당하여 이 토큰들을 파인튜닝으로 학습, 후에 Decoder 가 해당 토큰을 마주할 시 API의 답을 받아적고 다음 토큰으로 넘어가는 방식을 채택하였다. Toolformer(disabled)와의 비교를 통해 실제 API 호출의 효용을 확인했으며 GPT-J+CC 와의 비교를 통해 파인튜닝의 영향성이 없음을 확인하였다.

- 한계점 : 단, 이러한 파인튜닝 데이터셋의 구조적인 문제로 인해 연쇄적인 Tool 사용, 다양한 Tool 결과 사용, Tool 추가, 혹은 재질문 등이 불가능하다는 한계점을 가지고 있다. 또한 one-shot model의 구조적인 문제로 API 호출 시 출력 결과를 후처리하기 힘들다는 단점이 존재한다.

- 기능 : 사용된 Tool로는 Question Answering, Calculator, Wikipedia Search, Machine Translation System, Calender 기능이 존재하며 이들을 통해 약 10배 이상의 파라미터 수를 지닌 GPT-3 모델보다 더 나은 성능을 보일 수 있었다. 모델의 파라미터 수가 커짐에 따라 여전히 disabled과의 성능차가 보이는 것은 API의 활용능력이 늘어났기 때문임을 알 수 있었다.

- 평가 : LAMA(LAnguage Model Analysis, 본연의 언어 능력), Math Benchmarks, Multilingual QA(번역), QA, Temporal Datasets(시간적 데이터)등의 데이터셋을 통해 테스트가 이루어졌으며 다언어 지원과 일반 QA 를 제외하고는 GPT-3 보다 더 나은 성능을 보였다. Perplexity(혼잡도) 기준 파인튜닝이 본연의 언어 능력에 큰 감소를 불러일으키지 않았다.

- 후속연구 : Toolken

- 키워드 : 
    1) self-supervised learning : 모델 본인이 라벨없이 학습 데이터를 만들어 학습하는 과정.
            여기서는 special token이 포함된 passage를 본인이 [MASK] 로 잘라 문맥을 학습하는 과정을 말한다.
    2) in-context learning : ㅁㅇㄴ미ㅓㅏㄹㄴㅁ이ㅏ룸ㄴㅇ

### Retrieval-Augmentaed Generation for Knowledge-Intensive NLP Tasks
- 요약 : 현재 존재하는 LLM 은 외부 정보, 어느 기업 내부 정보 등을 학습하지 못했다는 정보의 한계 및 Hallucination이라는 치명적인 단점을 가지고 있다. 이에 외부 정보 요청 시 별도의 검색 후 답변하는 LLM 이 사용되어 왔는데 이 과정과 모델의 응답 생성 과정을 통합시키는 RAG 과정을 구현해내었다.

- 방법론 : DPR(Dense Passage Retriever)를 통해 사용자의 질문과 가장 높은 유사성을 가진 문서를 별도의 Document Index에서 MIPS, 최대 내적값을 통해 뽑아낸다. 이렇게 뽑은 문서(zi)와 이전 토큰들(y1~yj), 사용자의 질문(x)를 받아 다음 토큰(RAG-Token) 혹은 문장(RAG-Sequence) 의 확률을 계산해낸다. RAG-Sequence의 경우 특정 답변 y가 각 문서 zi 별로 나올 확률을 이전 토큰들을 고려하여 계산해낸 방식이며, RAG-Token의 경우 특정 토큰 yj가 각 문서 zi 별로 나올 확률을 이전 토큰들을 고려하여 계산한 방식이다. 이는 기존 parametic decoder와 non-parametic DPR을 통합한 시도로써 의미가 크다.

- 한계점 : 없음.

- 기능 : Retrieval Ablations(요소 제거법)을 통해 DPR의 성능 개선 명확성을 확보, 여러 Panelty 기법을 통해 Diversity를 확인하였으며 Index hot-swapping(최신 정보 유지)에서도 높은 수치를 기록하였다.

- 평가 : Open-domain question answering(QA)와 Abstractive QA(명확한 답변이 불가능한 Hallucination 유발), Jeopardy Question Generation(관계 추출, 추론 능력), FEVER(Fact Extraction and VERification, 사실 판단 능력) 을 바탕으로 open-book/closed-book 모델들을 대상으로 평가가 진행되었다. 

- 후속연구 : 여러 다양한 Retriever, HyDE 등의 기법

- 키워드 : 